# 2. Create KItems with the SDK

In this tutorial we see how to create new KItems.

### 2.1. Setting up
Before you run this tutorial: make sure to have access to a DSMS-instance of your interest, along with installation of this package, and have established access to the DSMS through DSMS-SDK (refer to [Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms))

Now let us import the needed classes and functions for this tutorial.

In [1]:
from dsms import DSMS, KItem

Now source the environmental variables from an `.env` file and start the DSMS-session.

In [2]:
import os
dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()


### 2.2. Create KItems

We can make new KItems by simple class-initiation. (Do not pass an existing KItem as input, as this will raise an error.)

In [3]:
item = KItem(
    name="Specimen123",
    ktype_id=dsms.ktypes.Specimen,
    custom_properties = {
        "Width": 0.5,
        "Length": 0.15,
    }
)

item

/root/dsms/dsms-python-sdk/dsms/knowledge/kitem.py:430: UserWarning: A flat dictionary was provided for custom properties.
                    Will be transformed into `KItemCustomPropertiesModel`.
  warnings.warn(


kitem:
  name: Specimen123
  ktype_id: specimen
  custom_properties:
    content:
      sections:
      - id: section-specimen-info
        name: Specimen Information
        entries:
        - id: input-specimen-width
          type: Number
          label: Width
          value: 0.5
          measurementUnit: null
          relationMapping: null
          required: false
        - id: input-specimen-length
          type: Number
          label: Length
          value: 0.15
          measurementUnit: null
          relationMapping: null
          required: false

Remember: changes are only synchronized with the DSMS when you call the `commit`-method:

In [4]:
dsms.add(item)
dsms.commit()
item.url

'https://nash.materials-data.space/knowledge/specimen/specimen123-227d1678'

As we can see, the object we created before running the `commit`-method has automatically been updated, e.g. with the creation- and update-timestamp. We can check this with the below command:

In [5]:
item

kitem:
  id: 227d1678-aa56-4598-9082-540c629fd8a8
  name: Specimen123
  ktype_id: specimen
  slug: specimen123-227d1678
  avatar_exists: false
  has_contexts: false
  annotations: []
  attachments: []
  linked_kitems: []
  affiliations: []
  authors: []
  contacts: []
  created_at: 2026-06-07 20:52:02.878207
  updated_at: 2026-06-07 20:52:02.878207
  external_links: []
  apps: []
  custom_properties:
    content:
      sections:
      - id: section-specimen-info
        name: Specimen Information
        entries:
        - id: input-specimen-width
          type: Number
          label: Width
          value: 0.5
          measurementUnit: null
          relationMapping: null
          required: false
        - id: input-specimen-length
          type: Number
          label: Length
          value: 0.15
          measurementUnit: null
          relationMapping: null
          required: false
  rdf_exists: false
  access_properties:
    visibility: private
    user_access:
    - role: 

To just get the name of the item, we can do it as follows:

In [6]:
item.name

'Specimen123'

As well as the id of the KItem we can do it as follows:

In [7]:
item.id

UUID('227d1678-aa56-4598-9082-540c629fd8a8')

To check the KType of the item newly created we can use the following:

In [8]:
item.ktype

ktype:
  id: specimen
  name: Specimen
  webform_schema_id: 7199e339-2512-4aed-8b99-af495c0349d3
  webform_schema:
    id: 7199e339-2512-4aed-8b99-af495c0349d3
    name: Specimen
    spec:
      semantics_enabled: true
      sections_enabled: false
      class_mapping: http://purl.obolibrary.org/obo/OBI_0100051
      sections:
      - id: section-specimen-info
        name: Specimen Information
        inputs:
        - id: input-specimen-type
          label: Specimen type
          widget: Text
          required: false
          hint: e.g. flat, round, notched
          hidden: false
          ignore: false
          select_options: []
          relation_mapping:
            iri: https://w3id.org/steel/ProcessOntology/hasSampleType_Object
            type: data_property
            inverse: false
          multiple_selection: false
        - id: input-specimen-geometry
          label: Specimen geometry
          widget: Text
          required: false
          hidden: false
       

... and also check the KType:

In [9]:
item.is_a(dsms.ktypes.Specimen)

True

We are able to print the subgraph related to the KItem:

In [10]:
try:
    print(item.subgraph.serialize())
except ValueError as e:
    print(f"Note: RDF subgraph is generated asynchronously.")
    print(f"It may not be available immediately after creation.")

Note: RDF subgraph is generated asynchronously.
It may not be available immediately after creation.


And we can convert the units:

In [11]:
try:
    item.custom_properties.Width.convert_to("m")
except ValueError as e:
    print(f"Unit conversion not available: {e}")

Unit conversion not available: Property `Width` does not own any
                unit with respect to the semantics applied.


In [12]:
try:
    item.custom_properties.Length.convert_to("m")
except ValueError as e:
    print(f"Unit conversion not available: {e}")

Unit conversion not available: Property `Length` does not own any
                unit with respect to the semantics applied.


You can also convert the custom_properties to a flat dict by passing the `flat`-parameter to the `model_dump`-method of the pydantic model:

In [13]:
item.custom_properties.model_dump(flat=True)

{'Width': 0.5, 'Length': 0.15}

### 2.x. Setting access properties

Access control is defined via `access_properties`, which assigns roles to specific users and groups. The available roles are `MEMBER` (read only), `CONTRIBUTOR` (read and update), and `OWNER` (read, update, delete, manage).

In [14]:
from dsms.knowledge.properties.access import KItemAccessProperties, Role

# Look up the current user to demonstrate role assignment
uname = dsms.config.username
if hasattr(uname, "get_secret_value"):
    uname = uname.get_secret_value()
current_user = dsms.users.by_username.get(uname)

item.access_properties = KItemAccessProperties(
    user_access=[{"user_id": current_user.id, "role": Role.OWNER}],
)
dsms.commit()

/root/dsms/dsms-python-sdk/dsms/core/dsms.py:222: UserWarning: Nothing to commit. No changes have been made to the DSMS instance.If you would like to add&/delete KItems, KTypes or AppConfigs,please use: `dsms.add(my_object)` or dsms.delete(my_object)`before running `dsms.commit()`.
  warnings.warn(



Now you can check if the particular KItem is in the list of KItems. This can be done either by using the command:
    `
     dsms.kitems
    `
    or by logging into the frontend dsms instance.